# Phase 4 — Leakage-Safe Feature Engineering

This notebook constructs model-ready features for the binary re-engagement prediction task.

The input dataset is the session-level modeling dataset created in Phase 3:
`model_dataset_100k_3day.csv`.

The prediction task is defined as binary classification:
given a user–streamer session at time `t`, predict whether the user will return to the same streamer within the next 3 days.

## Methodological constraints

To prevent data leakage, all engineered features must satisfy the following rule:

**Only information available strictly before the prediction timestamp may be used.**

This means:

- no future interactions may be used
- the current anchor session itself must not be included in historical feature values
- feature computation must respect the chronological train/validation/test design from Phase 3

## Planned feature groups

The first feature engineering version will focus on:

1. user history features
2. streamer history features
3. user–streamer interaction history features
4. temporal context features
5. simple structural proxy features

This notebook first implements a minimal but robust feature set before adding more advanced graph-oriented features in later iterations.

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
from collections import defaultdict

In [3]:
DATA_PATH = Path("../data_processed/model_dataset_100k_3day.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (2380571, 8)

Columns:
['user_id', 'streamer_name', 'stream_id', 'prediction_time', 'duration_intervals', 'duration_minutes', 'segment', 'label']


In [4]:
df.head()

,user_id,streamer_name,stream_id,prediction_time,duration_intervals,duration_minutes,segment,label
0,1,alptv,33846768288,166,3,30,train,0
1,1,berkriptepe,33903958784,734,3,30,train,0
2,1,elraenn,34079135968,2600,1,10,train,0
3,1,elraenn,34236660832,4314,1,10,validation,0
4,1,esl_csgo,34328509984,5208,1,10,test,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2380571 entries, 0 to 2380570
Data columns (total 8 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   user_id             int64 
 1   streamer_name       object
 2   stream_id           int64 
 3   prediction_time     int64 
 4   duration_intervals  int64 
 5   duration_minutes    int64 
 6   segment             object
 7   label               int64 
dtypes: int64(6), object(2)
memory usage: 145.3+ MB


In [6]:
df["segment"].value_counts(dropna=False)

segment
train         1373160
validation     520761
test           486650
Name: count, dtype: int64

## Chronological ordering and temporal consistency

All feature engineering must respect the temporal structure of the dataset.

The `prediction_time` column represents discrete time steps (10-minute intervals).

To ensure leakage-free feature computation:

- the dataset must be sorted by `prediction_time`
- historical features must be computed using only information available strictly before the current timestep
- sessions occurring at the same timestep must not influence one another's historical features
- no future information must be accessible during feature construction

This requires a timestep-level feature construction procedure:

1. **Compute stage**  
   For all rows with the same `prediction_time`, feature values are computed using only the historical state from earlier timesteps.

2. **Update stage**  
   After all rows at that timestep have received their feature values, the historical state is updated using the current timestep.

This guarantees strict temporal consistency and prevents within-timestep leakage.

In [7]:
df = df.sort_values("prediction_time").reset_index(drop=True)

print("Sorted by prediction_time.")
df[["prediction_time"]].head()

Sorted by prediction_time.


,prediction_time
0,0
1,0
2,0
3,0
4,0


In [8]:
# Monotonicity check
is_sorted = (df["prediction_time"].diff().fillna(0) >= 0).all()
print("Is globally sorted:", is_sorted)

Is globally sorted: True


In [9]:
# Segment order check
df.groupby("segment")["prediction_time"].agg(["min", "max"])

,min,max
segment,,
test,4752,5715
train,0,2879
validation,3312,4319


In [10]:
# User-level ordering check
sample_users = df["user_id"].drop_duplicates().sample(5, random_state=42)

for u in sample_users:
    user_times = df[df["user_id"] == u]["prediction_time"]
    is_user_sorted = (user_times.diff().fillna(0) >= 0).all()
    print(f"User {u} sorted:", is_user_sorted)

User 96675 sorted: True
User 80188 sorted: True
User 10416 sorted: True
User 66833 sorted: True
User 5476 sorted: True


## Feature taxonomy

To ensure a structured and interpretable feature engineering process, features are organized into conceptually distinct groups based on the type of information they represent.

This taxonomy is designed to be model-agnostic, ensuring that all models are trained on the same underlying information and enabling a fair comparison between traditional machine learning and graph-based approaches.

---

### 1. User history features

User history features capture the historical behavior of users prior to the prediction timestamp.

Examples:
- number of past sessions (`user_past_sessions`)
- total watch time (`user_total_watch_time`)
- average session duration (`user_avg_session_duration`)
- recency of last activity (`user_recency`)
- history indicator (`user_has_history`)

These features represent user-level engagement patterns and reflect how active, consistent, and recently engaged a user has been.

---

### 2. Streamer history features

Streamer history features capture the historical characteristics of streamers based on past interactions.

Examples:
- number of past sessions involving the streamer (`streamer_past_sessions`)
- total accumulated watch time (`streamer_total_watch_time`)
- average session duration (`streamer_avg_session_duration`)
- recency of viewer activity (`streamer_recency`)
- history indicator (`streamer_has_history`)

These features approximate streamer popularity, activity level, and temporal dynamics of audience engagement.

---

### 3. User–streamer interaction features

Interaction features capture the historical relationship between a specific user and streamer.

Examples:
- number of past interactions (`pair_past_sessions`)
- total watch time between user and streamer (`pair_total_watch_time`)
- average session duration (`pair_avg_session_duration`)
- time since last interaction (`pair_recency`)
- history indicator (`pair_has_history`)

These features represent the strength and recency of the relationship between a user and a streamer and are expected to be strong predictors of re-engagement.

---

### 4. Temporal context features

Temporal features capture time-dependent patterns in user behavior.

The dataset is based on discrete timesteps (10-minute intervals) and does not include absolute timestamps (e.g., real-world time of day or calendar dates). Therefore, temporal context is represented using cyclic transformations of the timestep index.

#### Cyclic temporal encoding

To model periodic behavior, sine and cosine transformations are applied at multiple time scales:

- Daily cycle (144 timesteps): captures within-day activity patterns  
- Weekly cycle (1008 timesteps): captures longer-term engagement patterns  

These features enable the model to learn periodic behavioral patterns without relying on external time information.

---

### 5. Structural graph-based features

Structural features incorporate information from the underlying bipartite user–stream interaction graph into the tabular modeling pipeline.

Rather than computing full graph embeddings, these features approximate graph structure using leakage-safe statistics derived from historical interactions.

#### Node-level structural features

Node-level features describe the structural position of users and streamers in the interaction graph.

For users:
- number of unique streamers interacted with (`user_unique_streamers`)
- repeat interaction ratio (`user_repeat_ratio`), defined as the ratio of total interactions to unique streamers

For streamers:
- number of unique users (`streamer_unique_users`)
- repeat audience ratio (`streamer_repeat_audience_ratio`), defined as the ratio of total interactions to unique users

These features approximate node degree and interaction concentration, capturing whether behavior is exploratory or repetitive.

#### Pair-level structural features

Pair-level features capture the relative importance of a user–streamer relationship within the broader interaction graph.

- `pair_watch_share_user`: proportion of the user’s total watch time spent on the current streamer  
- `pair_watch_share_streamer`: proportion of the streamer’s total watch time contributed by the current user  

These features approximate normalized edge strength and provide a measure of interaction intensity relative to the user’s and streamer’s overall activity.

---

### Methodological note

All features are computed in a strictly chronological and leakage-free manner.

For each timestep:

- feature values are computed using only information available from earlier timesteps  
- sessions occurring within the same timestep do not influence one another  
- historical state is updated only after all rows at the current timestep have been processed  

This ensures that all features reflect information available at prediction time and that the evaluation setup remains consistent with real-world deployment conditions.

---

### Handling of recency features

For recency-based features, the value is defined only when prior history exists.

If no prior interaction is available, recency is recorded as missing (`NaN`).  
These missing values are structurally meaningful and indicate first-time observations rather than data quality issues.

To preserve this distinction, dedicated binary history indicators are included:

- `user_has_history`
- `streamer_has_history`
- `pair_has_history`

This design allows downstream models to explicitly distinguish between missing history and short recency intervals.

## User history feature construction

In [11]:
# Historical state containers
user_session_count = defaultdict(int)
user_total_watch_time_state = defaultdict(int)
user_last_time = {}

# Output feature lists
user_past_sessions = []
user_total_watch_time = []
user_avg_session_duration = []
user_recency = []
user_has_history = []

# Timestep-level computation to prevent within-timestep leakage
for t, group in df.groupby("prediction_time", sort=True):
    group_rows = list(group.itertuples(index=False))

    # ----- compute features from strictly past timesteps only -----
    for row in group_rows:
        user = row.user_id

        past_sessions = user_session_count[user]
        total_watch_time = user_total_watch_time_state[user]

        if past_sessions > 0:
            avg_duration = total_watch_time / past_sessions
            recency = t - user_last_time[user]
            has_history = 1
        else:
            avg_duration = 0
            recency = np.nan
            has_history = 0

        user_past_sessions.append(past_sessions)
        user_total_watch_time.append(total_watch_time)
        user_avg_session_duration.append(avg_duration)
        user_recency.append(recency)
        user_has_history.append(has_history)

    # ----- update historical state only after all rows at timestep t are processed -----
    for row in group_rows:
        user = row.user_id
        duration = row.duration_minutes

        user_session_count[user] += 1
        user_total_watch_time_state[user] += duration
        user_last_time[user] = t

df["user_past_sessions"] = user_past_sessions
df["user_total_watch_time"] = user_total_watch_time
df["user_avg_session_duration"] = user_avg_session_duration
df["user_recency"] = user_recency
df["user_has_history"] = user_has_history

## User history feature sanity checks

The first rows and summary statistics are inspected to verify that:

- first-time users receive zero-history values
- historical counts increase over time
- recency is defined only when prior user history exists
- the binary history indicator correctly distinguishes first-time from repeat observations

In [12]:
df[[
    "user_id",
    "prediction_time",
    "duration_minutes",
    "user_past_sessions",
    "user_total_watch_time",
    "user_avg_session_duration",
    "user_recency",
    "user_has_history"
]].head(10)

,user_id,prediction_time,duration_minutes,user_past_sessions,user_total_watch_time,user_avg_session_duration,user_recency,user_has_history
0,847,0,10,0,0,0.0,NaN,0
1,86110,0,50,0,0,0.0,NaN,0
2,95292,0,20,0,0,0.0,NaN,0
3,80024,0,10,0,0,0.0,NaN,0
4,76125,0,20,0,0,0.0,NaN,0
5,63098,0,70,0,0,0.0,NaN,0
6,63091,0,20,0,0,0.0,NaN,0
7,99254,0,40,0,0,0.0,NaN,0
8,82628,0,10,0,0,0.0,NaN,0
9,54789,0,20,0,0,0.0,NaN,0


In [13]:
len(user_past_sessions), len(df)

(2380571, 2380571)

In [14]:
df["user_past_sessions"].describe()

count    2.380571e+06
mean     2.655477e+01
std      2.828382e+01
min      0.000000e+00
25%      6.000000e+00
50%      1.700000e+01
75%      3.800000e+01
max      2.870000e+02
Name: user_past_sessions, dtype: float64

In [15]:
df["user_recency"].describe()

count    2.261017e+06
mean     1.811771e+02
std      3.713896e+02
min      1.000000e+00
25%      8.000000e+00
50%      5.700000e+01
75%      1.520000e+02
max      5.551000e+03
Name: user_recency, dtype: float64

In [16]:
df["user_has_history"].value_counts(dropna=False)

user_has_history
1    2261017
0     119554
Name: count, dtype: int64

In [17]:
df["user_recency"].isnull().mean()

0.050220724355627286

## Streamer history feature construction

In [18]:
# state
streamer_session_count = defaultdict(int)
streamer_total_watch_time_state = defaultdict(int)
streamer_last_time = {}

# outputs
streamer_past_sessions = []
streamer_total_watch_time = []
streamer_avg_session_duration = []
streamer_recency = []
streamer_has_history = []

# timestep-level computation
for t, group in df.groupby("prediction_time", sort=True):
    group_rows = list(group.itertuples(index=False))

    # ----- compute from strictly past timesteps only -----
    for row in group_rows:
        streamer = row.streamer_name

        past_sessions = streamer_session_count[streamer]
        total_time = streamer_total_watch_time_state[streamer]

        if past_sessions > 0:
            avg_duration = total_time / past_sessions
            recency = t - streamer_last_time[streamer]
            has_history = 1
        else:
            avg_duration = 0
            recency = np.nan
            has_history = 0

        streamer_past_sessions.append(past_sessions)
        streamer_total_watch_time.append(total_time)
        streamer_avg_session_duration.append(avg_duration)
        streamer_recency.append(recency)
        streamer_has_history.append(has_history)

    # ----- update AFTER full timestep -----
    for row in group_rows:
        streamer = row.streamer_name
        duration = row.duration_minutes

        streamer_session_count[streamer] += 1
        streamer_total_watch_time_state[streamer] += duration
        streamer_last_time[streamer] = t

df["streamer_past_sessions"] = streamer_past_sessions
df["streamer_total_watch_time"] = streamer_total_watch_time
df["streamer_avg_session_duration"] = streamer_avg_session_duration
df["streamer_recency"] = streamer_recency
df["streamer_has_history"] = streamer_has_history

## Streamer history feature sanity checks

The first rows and summary statistics are inspected to verify that:

- first-time streamers receive zero-history values
- historical counts increase over time
- recency is defined only when prior streamer history exists
- the binary history indicator correctly distinguishes first-time from repeated streamer activity

In [19]:
df[[
    "streamer_name",
    "prediction_time",
    "streamer_past_sessions",
    "streamer_total_watch_time",
    "streamer_avg_session_duration",
    "streamer_recency",
    "streamer_has_history"
]].head(10)

,streamer_name,prediction_time,streamer_past_sessions,streamer_total_watch_time,streamer_avg_session_duration,streamer_recency,streamer_has_history
0,catsonurhead,0,0,0,0.0,NaN,0
1,riverfoxtv,0,0,0,0.0,NaN,0
2,solaryfortnite,0,0,0,0.0,NaN,0
3,risenhaha,0,0,0,0.0,NaN,0
4,blondynkitezgraja,0,0,0,0.0,NaN,0
5,trymacs,0,0,0,0.0,NaN,0
6,trymacs,0,0,0,0.0,NaN,0
7,kinggothalion,0,0,0,0.0,NaN,0
8,hiko,0,0,0,0.0,NaN,0
9,dmbrandon,0,0,0,0.0,NaN,0


In [20]:
len(streamer_past_sessions), len(df)

(2380571, 2380571)

In [21]:
df["streamer_past_sessions"].describe()

count    2.380571e+06
mean     1.283731e+03
std      3.502956e+03
min      0.000000e+00
25%      1.600000e+01
50%      1.520000e+02
75%      8.600000e+02
max      3.598500e+04
Name: streamer_past_sessions, dtype: float64

In [22]:
df["streamer_recency"].describe()

count    2.225591e+06
mean     1.096436e+02
std      3.847715e+02
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      1.100000e+01
max      5.683000e+03
Name: streamer_recency, dtype: float64

In [23]:
df["streamer_has_history"].value_counts(dropna=False)

streamer_has_history
1    2225591
0     154980
Name: count, dtype: int64

In [24]:
df["streamer_recency"].isnull().mean()

0.06510202804285191

## User–streamer interaction feature construction

In [25]:
# state
pair_session_count = defaultdict(int)
pair_total_watch_time_state = defaultdict(int)
pair_last_time = {}

# outputs
pair_past_sessions = []
pair_total_watch_time = []
pair_avg_session_duration = []
pair_recency = []
pair_has_history = []

# timestep-level computation
for t, group in df.groupby("prediction_time", sort=True):
    group_rows = list(group.itertuples(index=False))

    # ----- compute from strictly past timesteps only -----
    for row in group_rows:
        user = row.user_id
        streamer = row.streamer_name
        key = (user, streamer)

        past_sessions = pair_session_count[key]
        total_time = pair_total_watch_time_state[key]

        if past_sessions > 0:
            avg_duration = total_time / past_sessions
            recency = t - pair_last_time[key]
            has_history = 1
        else:
            avg_duration = 0
            recency = np.nan
            has_history = 0

        pair_past_sessions.append(past_sessions)
        pair_total_watch_time.append(total_time)
        pair_avg_session_duration.append(avg_duration)
        pair_recency.append(recency)
        pair_has_history.append(has_history)

    # ----- update AFTER full timestep -----
    for row in group_rows:
        user = row.user_id
        streamer = row.streamer_name
        duration = row.duration_minutes
        key = (user, streamer)

        pair_session_count[key] += 1
        pair_total_watch_time_state[key] += duration
        pair_last_time[key] = t

df["pair_past_sessions"] = pair_past_sessions
df["pair_total_watch_time"] = pair_total_watch_time
df["pair_avg_session_duration"] = pair_avg_session_duration
df["pair_recency"] = pair_recency
df["pair_has_history"] = pair_has_history

## User–streamer interaction feature sanity checks

The first rows and summary statistics are inspected to verify that:

- first-time user–streamer pairs receive zero-history values
- interaction counts increase over time for repeated pairs
- recency is defined only when prior pair history exists
- the binary history indicator correctly distinguishes first-time from repeated pair interactions

In [26]:
df[[
    "user_id",
    "streamer_name",
    "prediction_time",
    "pair_past_sessions",
    "pair_total_watch_time",
    "pair_avg_session_duration",
    "pair_recency",
    "pair_has_history"
]].head(10)

,user_id,streamer_name,prediction_time,pair_past_sessions,pair_total_watch_time,pair_avg_session_duration,pair_recency,pair_has_history
0,847,catsonurhead,0,0,0,0.0,NaN,0
1,86110,riverfoxtv,0,0,0,0.0,NaN,0
2,95292,solaryfortnite,0,0,0,0.0,NaN,0
3,80024,risenhaha,0,0,0,0.0,NaN,0
4,76125,blondynkitezgraja,0,0,0,0.0,NaN,0
5,63098,trymacs,0,0,0,0.0,NaN,0
6,63091,trymacs,0,0,0,0.0,NaN,0
7,99254,kinggothalion,0,0,0,0.0,NaN,0
8,82628,hiko,0,0,0,0.0,NaN,0
9,54789,dmbrandon,0,0,0,0.0,NaN,0


In [27]:
len(pair_past_sessions), len(df)

(2380571, 2380571)

In [28]:
df["pair_past_sessions"].describe()

count    2.380571e+06
mean     1.282296e+00
std      2.239866e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.000000e+00
max      7.400000e+01
Name: pair_past_sessions, dtype: float64

In [29]:
df["pair_recency"].describe()

count    1.101181e+06
mean     9.327905e+02
std      9.847609e+02
min      1.000000e+00
25%      1.860000e+02
50%      5.770000e+02
75%      1.295000e+03
max      5.714000e+03
Name: pair_recency, dtype: float64

In [30]:
df["pair_has_history"].value_counts(dropna=False)

pair_has_history
0    1279390
1    1101181
Name: count, dtype: int64

In [31]:
df["pair_recency"].isnull().mean()

0.5374298855190625

## Temporal context features

To capture daily periodic patterns, the timestep variable is transformed using sine and cosine functions. This encoding allows models to learn cyclic behavior without introducing artificial boundaries in time representation.

In [32]:
TIMESTEPS_PER_DAY = 144
TIMESTEPS_PER_WEEK = 144 * 7

# --- Daily cycle features ---
# captures within-day periodic patterns (e.g., activity peaks during certain hours)
df["time_sin_day"] = np.sin(2 * np.pi * df["prediction_time"] / TIMESTEPS_PER_DAY)
df["time_cos_day"] = np.cos(2 * np.pi * df["prediction_time"] / TIMESTEPS_PER_DAY)

# --- Weekly cycle features ---
# captures longer-term periodic patterns across days (e.g., weekday vs weekend-like behavior)
df["time_sin_week"] = np.sin(2 * np.pi * df["prediction_time"] / TIMESTEPS_PER_WEEK)
df["time_cos_week"] = np.cos(2 * np.pi * df["prediction_time"] / TIMESTEPS_PER_WEEK)

## Temporal context features sanity checks

In [33]:
df[[
    "time_sin_day", "time_cos_day",
    "time_sin_week", "time_cos_week"
]].describe()

,time_sin_day,time_cos_day,time_sin_week,time_cos_week
count,2.380571e+06,2.380571e+06,2.380571e+06,2.380571e+06
mean,-8.852526e-02,1.672045e-01,3.341956e-02,-2.767520e-02
std,6.989101e-01,6.897325e-01,7.054463e-01,7.074342e-01
min,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00
25%,-7.660444e-01,-4.617486e-01,-6.892583e-01,-7.700362e-01
50%,-1.736482e-01,3.007058e-01,6.229283e-02,-8.715574e-02
75%,6.087614e-01,8.191520e-01,7.158668e-01,6.616858e-01
max,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


In [34]:
df[[
    "time_sin_day", "time_cos_day",
    "time_sin_week", "time_cos_week"
]].isnull().sum()

time_sin_day     0
time_cos_day     0
time_sin_week    0
time_cos_week    0
dtype: int64

In [35]:
np.isfinite(df[[
    "time_sin_day", "time_cos_day",
    "time_sin_week", "time_cos_week"
]]).all()

time_sin_day     True
time_cos_day     True
time_sin_week    True
time_cos_week    True
dtype: bool

In [36]:
df[[
    "prediction_time",
    "time_sin_day",
    "time_cos_day"
]].head(200)

,prediction_time,time_sin_day,time_cos_day
0,0,0.0,1.0
1,0,0.0,1.0
2,0,0.0,1.0
3,0,0.0,1.0
4,0,0.0,1.0
...,...,...,...
195,0,0.0,1.0
196,0,0.0,1.0
197,0,0.0,1.0
198,0,0.0,1.0


In [37]:
df[[
    "time_sin_day", "time_cos_day",
    "time_sin_week", "time_cos_week"
]].corr()

,time_sin_day,time_cos_day,time_sin_week,time_cos_week
time_sin_day,1.000000,0.029126,-0.002677,-0.013460
time_cos_day,0.029126,1.000000,-0.005335,0.006375
time_sin_week,-0.002677,-0.005335,1.000000,0.018388
time_cos_week,-0.013460,0.006375,0.018388,1.000000


## Structural graph-based features

In addition to behavioral and temporal features, structural properties of the user–stream interaction graph are incorporated into the tabular modeling pipeline.

The interaction data naturally forms a bipartite graph between users and streamers. Instead of relying on full graph-based learning, we derive leakage-safe structural features that approximate graph properties using historical interaction data.

These features are computed at each prediction timestep using only information available from earlier timesteps, ensuring strict temporal consistency.

### Node-level structural features

Node-level features capture the structural position of users and streamers in the interaction graph.

For users:
- **user_unique_streamers**: number of distinct streamers the user has interacted with in the past (bipartite degree)
- **user_repeat_ratio**: ratio between total past interactions and unique streamers, indicating whether the user repeatedly interacts with a small set of streamers or explores many different ones

For streamers:
- **streamer_unique_users**: number of distinct users who have interacted with the streamer (degree)
- **streamer_repeat_audience_ratio**: ratio between total interactions and unique users, capturing the extent to which the streamer’s audience consists of repeat viewers

These features provide a lightweight approximation of node degree and interaction concentration in the graph.

### Pair-level structural features

Pair-level features capture the strength of the relationship between a specific user and streamer.

- **pair_watch_share_user**: proportion of the user's total watch time that is spent on the current streamer
- **pair_watch_share_streamer**: proportion of the streamer's total watch time contributed by the current user

These features represent normalized interaction intensity and approximate edge strength in the bipartite graph.

### Methodological note

All structural features are computed using a timestep-level procedure:

- feature values are computed using only historical interactions from earlier timesteps
- interactions within the same timestep do not influence one another
- structural state (e.g., seen users or streamers) is updated only after processing all rows at the current timestep

This ensures that all structural features are leakage-free and consistent with the temporal prediction setting.

### Interpretation

These structural features act as a bridge between traditional machine learning models and graph-based approaches. They allow the analysis of how graph structure contributes to predictive performance without requiring full graph neural network representations.

In [38]:
# Structural graph-based features
# These features approximate node degree and normalized edge strength
# in the historical user–stream interaction graph.

# Historical structural state
user_seen_streamers = defaultdict(set)
streamer_seen_users = defaultdict(set)

# Output feature lists
user_unique_streamers = []
streamer_unique_users = []
user_repeat_ratio = []
streamer_repeat_audience_ratio = []
pair_watch_share_user = []
pair_watch_share_streamer = []

# Timestep-level computation
for t, group in df.groupby("prediction_time", sort=True):
    group_rows = list(group.itertuples(index=False))

    # ----- compute structural features from strictly past timesteps only -----
    for row in group_rows:
        user = row.user_id
        streamer = row.streamer_name

        # node-level degree features
        user_deg = len(user_seen_streamers[user])
        streamer_deg = len(streamer_seen_users[streamer])

        user_unique_streamers.append(user_deg)
        streamer_unique_users.append(streamer_deg)

        # normalized repeat / concentration features
        user_repeat_ratio.append(row.user_past_sessions / (user_deg + 1))
        streamer_repeat_audience_ratio.append(row.streamer_past_sessions / (streamer_deg + 1))

        # normalized pair watch-time shares
        pair_watch_share_user.append(row.pair_total_watch_time / (row.user_total_watch_time + 1))
        pair_watch_share_streamer.append(row.pair_total_watch_time / (row.streamer_total_watch_time + 1))

    # ----- update structural state only after full timestep is processed -----
    for row in group_rows:
        user = row.user_id
        streamer = row.streamer_name

        user_seen_streamers[user].add(streamer)
        streamer_seen_users[streamer].add(user)

df["user_unique_streamers"] = user_unique_streamers
df["streamer_unique_users"] = streamer_unique_users
df["user_repeat_ratio"] = user_repeat_ratio
df["streamer_repeat_audience_ratio"] = streamer_repeat_audience_ratio
df["pair_watch_share_user"] = pair_watch_share_user
df["pair_watch_share_streamer"] = pair_watch_share_streamer

### Structural graph-based feature sanity checks

The structural features are inspected to verify that:

- node-level degree features are zero for first-time users or streamers
- repeat ratios remain well-defined for low-history cases
- pair watch-share features remain bounded and interpretable
- structural state is derived only from past timesteps

In [39]:
df[[
    "user_id",
    "streamer_name",
    "prediction_time",
    "user_unique_streamers",
    "streamer_unique_users",
    "user_repeat_ratio",
    "streamer_repeat_audience_ratio",
    "pair_watch_share_user",
    "pair_watch_share_streamer"
]].head(10)

,user_id,streamer_name,prediction_time,user_unique_streamers,streamer_unique_users,user_repeat_ratio,streamer_repeat_audience_ratio,pair_watch_share_user,pair_watch_share_streamer
0,847,catsonurhead,0,0,0,0.0,0.0,0.0,0.0
1,86110,riverfoxtv,0,0,0,0.0,0.0,0.0,0.0
2,95292,solaryfortnite,0,0,0,0.0,0.0,0.0,0.0
3,80024,risenhaha,0,0,0,0.0,0.0,0.0,0.0
4,76125,blondynkitezgraja,0,0,0,0.0,0.0,0.0,0.0
5,63098,trymacs,0,0,0,0.0,0.0,0.0,0.0
6,63091,trymacs,0,0,0,0.0,0.0,0.0,0.0
7,99254,kinggothalion,0,0,0,0.0,0.0,0.0,0.0
8,82628,hiko,0,0,0,0.0,0.0,0.0,0.0
9,54789,dmbrandon,0,0,0,0.0,0.0,0.0,0.0


In [40]:
df[[
    "user_unique_streamers",
    "streamer_unique_users",
    "user_repeat_ratio",
    "streamer_repeat_audience_ratio",
    "pair_watch_share_user",
    "pair_watch_share_streamer"
]].describe()

,user_unique_streamers,streamer_unique_users,user_repeat_ratio,streamer_repeat_audience_ratio,pair_watch_share_user,pair_watch_share_streamer
count,2.380571e+06,2.380571e+06,2.380571e+06,2.380571e+06,2.380571e+06,2.380571e+06
mean,1.519606e+01,6.913333e+02,1.369373e+00,1.449123e+00,8.220663e-02,4.624932e-02
std,1.466397e+01,1.741431e+03,6.670685e-01,7.189183e-01,1.780088e-01,1.704872e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,4.000000e+00,1.200000e+01,1.000000e+00,1.071429e+00,0.000000e+00,0.000000e+00
50%,1.100000e+01,1.010000e+02,1.333333e+00,1.417266e+00,0.000000e+00,0.000000e+00
75%,2.200000e+01,5.010000e+02,1.750000e+00,1.826087e+00,7.481297e-02,4.243663e-03
max,2.510000e+02,1.540800e+04,2.150000e+01,3.050000e+01,9.997960e-01,9.996826e-01


## Feature cleaning & final dataset

In [41]:
FEATURE_COLUMNS = [
    # user
    "user_past_sessions",
    "user_total_watch_time",
    "user_avg_session_duration",
    "user_recency",
    "user_has_history",

    # streamer
    "streamer_past_sessions",
    "streamer_total_watch_time",
    "streamer_avg_session_duration",
    "streamer_recency",
    "streamer_has_history",

    # interaction
    "pair_past_sessions",
    "pair_total_watch_time",
    "pair_avg_session_duration",
    "pair_recency",
    "pair_has_history",

    # temporal
    "time_sin_day",
    "time_cos_day",
    "time_sin_week",
    "time_cos_week",

    # structural
    "user_unique_streamers",
    "streamer_unique_users",
    "user_repeat_ratio",
    "streamer_repeat_audience_ratio",
    "pair_watch_share_user",
    "pair_watch_share_streamer",
]

In [42]:
df_model = df[["user_id", "streamer_name", "prediction_time"] + FEATURE_COLUMNS + ["label", "segment"]].copy()

In [43]:
print("Shape:", df_model.shape)

print("\nMissing values by column:")
print(df_model.isnull().sum().sort_values(ascending=False).head(10))

print("\nTotal missing values:", df_model.isnull().sum().sum())

Shape: (2380571, 30)

Missing values by column:
pair_recency                 1279390
streamer_recency              154980
user_recency                  119554
prediction_time                    0
user_id                            0
user_past_sessions                 0
user_avg_session_duration          0
user_total_watch_time              0
streamer_past_sessions             0
user_has_history                   0
dtype: int64

Total missing values: 1553924


In [44]:
OUTPUT_PATH = "../data_processed/model_dataset_100k_3day_features_v1.csv"

df_model.to_csv(OUTPUT_PATH, index=False)

In [45]:
df_model.head()

,user_id,streamer_name,prediction_time,user_past_sessions,user_total_watch_time,user_avg_session_duration,user_recency,user_has_history,streamer_past_sessions,streamer_total_watch_time,...,time_sin_week,time_cos_week,user_unique_streamers,streamer_unique_users,user_repeat_ratio,streamer_repeat_audience_ratio,pair_watch_share_user,pair_watch_share_streamer,label,segment
0,847,catsonurhead,0,0,0,0.0,NaN,0,0,0,...,0.0,1.0,0,0,0.0,0.0,0.0,0.0,0,train
1,86110,riverfoxtv,0,0,0,0.0,NaN,0,0,0,...,0.0,1.0,0,0,0.0,0.0,0.0,0.0,0,train
2,95292,solaryfortnite,0,0,0,0.0,NaN,0,0,0,...,0.0,1.0,0,0,0.0,0.0,0.0,0.0,1,train
3,80024,risenhaha,0,0,0,0.0,NaN,0,0,0,...,0.0,1.0,0,0,0.0,0.0,0.0,0.0,1,train
4,76125,blondynkitezgraja,0,0,0,0.0,NaN,0,0,0,...,0.0,1.0,0,0,0.0,0.0,0.0,0.0,0,train


In [46]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2380571 entries, 0 to 2380570
Data columns (total 30 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   user_id                         int64  
 1   streamer_name                   object 
 2   prediction_time                 int64  
 3   user_past_sessions              int64  
 4   user_total_watch_time           int64  
 5   user_avg_session_duration       float64
 6   user_recency                    float64
 7   user_has_history                int64  
 8   streamer_past_sessions          int64  
 9   streamer_total_watch_time       int64  
 10  streamer_avg_session_duration   float64
 11  streamer_recency                float64
 12  streamer_has_history            int64  
 13  pair_past_sessions              int64  
 14  pair_total_watch_time           int64  
 15  pair_avg_session_duration       float64
 16  pair_recency                    float64
 17  pair_has_history           

### Phase 4 summary

- Leakage-safe feature engineering implemented using a timestep-level computation framework  
- User, streamer, and user–streamer interaction features constructed from historical behavior  
- Temporal context incorporated via cyclic encoding of timestep-based periodic patterns  
- Structural graph-based features derived, including node-level degree proxies and normalized interaction intensity measures  
- All features computed using strictly past information, preventing both temporal and within-timestep leakage  
- Recency features are recorded as missing when no prior history exists, with dedicated binary indicators added to preserve this distinction  
- Final modeling dataset: ~2.38 million sessions with 25 engineered features capturing behavioral, temporal, and structural information